# Chuẩn bị dữ liệu
Gọi scripts/prepare_data.py và scripts/gen_noise.py (cùng các lệnh trong README, mục *Chạy lại từ đầu* và *Ablation mật độ nhiễu*). Phụ trách: TV1.

Kết quả nằm trong `data/processed/`: VSEC-dev/test, Viwiki-test, câu sạch cho LM và Syn-train/val/test (seed 42/43/44). Notebook 03 (n-gram), `scripts/predict.py`/`evaluate.py` và notebook 04 (BARTpho) đều đọc từ thư mục này.

In [ ]:
# Colab/Kaggle: clone repo và cài đặt
# !git clone <URL_REPO> && cd <TEN_REPO>
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# Dữ liệu thật + câu sạch (LM-corpus, clean_syn_{train,val,test}.txt)
!python scripts/prepare_data.py --config configs/data.yaml
# Syn-train / Syn-val / Syn-test: seed khác nhau như README
!python scripts/gen_noise.py --input data/processed/clean_syn_train.txt --output data/processed/syn_train.jsonl
!python scripts/gen_noise.py --input data/processed/clean_syn_val.txt   --output data/processed/syn_val.jsonl  --seed 43
!python scripts/gen_noise.py --input data/processed/clean_syn_test.txt  --output data/processed/syn_test.jsonl --seed 44

In [ ]:
# (Tùy chọn) dữ liệu cho ablation C: mật độ nhiễu 5/12/25% và v4 (nhiễu dấu), 50k câu, seed 42
RUN_ABLATION_DATA = False
if RUN_ABLATION_DATA:
    for tag, rate in [("d05", 0.05), ("d12", 0.12), ("d25", 0.25)]:
        !python scripts/gen_noise.py --input data/processed/clean_syn_train.txt --limit 50000 --error-rate {rate} --seed 42 --output data/processed/syn_train_{tag}.jsonl
    !python scripts/gen_noise.py --input data/processed/clean_syn_train.txt --config configs/noise_v4.yaml --limit 50000 --output data/processed/syn_train_v4.jsonl

In [ ]:
# Kiểm tra đủ đầu vào cho các bước sau
import os
need = ["vsec_dev", "vsec_test", "viwiki_test", "syn_train", "syn_val", "syn_test"]
missing = [n for n in need if not os.path.exists(f"data/processed/{n}.jsonl")]
assert not missing, f"Thiếu data/processed/{missing}.jsonl"
!wc -l data/processed/*.jsonl

## Chuyển dữ liệu sang môi trường GPU (trước notebook 04)
Notebook 04 chạy trên Kaggle/Colab nên không thấy `data/processed/` của máy này. Đóng gói rồi upload **mọi** file trong `data/kaggle_upload/` (các phần `*.partNN.jsonl.gz` và manifest `*.parts.json`) thành Kaggle dataset; notebook 04 tự kiểm tra đủ phần và ghép lại bằng `scripts/unpack_parts.py`.

In [ ]:
!python scripts/pack_for_kaggle.py --out data/kaggle_upload